In [ ]:
import zipfile, io
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
DATA = Path("data")


if not (DATA / "sigwatch_data").exists():
    for z in ["ngo_data_bocconi.zip", "banks_data_bocconi.zip"]:
        zipfile.ZipFile(DATA / z).extractall(DATA)


for p in sorted(DATA.rglob("*")):
    print(p)

In [ ]:
# Sigwatch
files = sorted((DATA / "sigwatch_data").glob("*.dta"))

sig = pd.concat(
    [pd.read_stata(f).assign(year_file=int(f.name[:4])) for f in files],
    ignore_index=True,
)
print(sig.shape)

In [ ]:
#Datastream


ri_sheets = pd.read_excel(DATA / "banks_data_bocconi" / "banks_ri.xlsm", sheet_name=None)
ri_sheets.pop("REQUEST_TABLE")

print(ri_sheets.keys())
print(ri_sheets["US"].shape)

In [ ]:
longs = []

for k, v in ri_sheets.items():
    long = v.melt(id_vars="Name", var_name="bank", value_name="ri")
    long = long.rename(columns={"Name": "date"})
    long["country"] = k
    longs.append(long)

ri = pd.concat(longs, ignore_index=True)

In [ ]:
print(ri.shape)
print(ri["country"].value_counts())
print(ri["bank"].nunique())

## 1. Sigwatch data

In [ ]:
print(f"{sig['uid_archive'].nunique():,} events, {len(sig):,} rows")
print(f"{sig['company'].nunique():,} companies in total") #nbr of unique company_name -> Number of companies
print(f"Average : {len(sig) / sig['uid_archive'].nunique():.1f} companies per event")


**Largest event:**

In [ ]:
top_uid = sig["uid_archive"].value_counts().index[0]
top = sig[sig["uid_archive"] == top_uid]
print(f"Largest event: {len(top)} firms named by {top['ngo_name1'].iloc[0]} on {top['date'].iloc[0]}")

Largest event. The event with the most targets is a single communication by ActionAid UK on 13 May 2013, naming 72 companies at once (BP, Shell, Tesco, AstraZeneca, …).

This matters for the event study: an NGO communication that names dozens of firms is unlikely to be firm-specific news, and any common price move that day would be absorbed by the market factor. We therefore compute n_targets (number of firms named in the same event) and will either drop or analyse separately the mass events above a threshold.

In [ ]:
counts = sig["uid_archive"].value_counts()          #number of firms per event
sig["n_targets"] = sig["uid_archive"].map(counts)   #attach it to every (event, firm) row

In [ ]:
events = sig.drop_duplicates("uid_archive")

print("n_targets, per (event, firm) row:")
print(sig["n_targets"].describe().round(2))

print("\nn_targets, per event:")
print(events["n_targets"].describe().round(2))

share_single = (events["n_targets"] == 1).mean()
print(f"\n{share_single:.1%} of events target a single company")